# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

# TODO : Enregistrer votre table Silver principale sous forme de vue temporaire SQL `v_silver`
SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

Lignes disponibles dans v_silver : 5819079


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [3]:
# TODO : Calculer moyenne, écart-type, skewness et quantiles (P25, P50, P75, P90, P95) en Spark SQL
df_dist_summary = spark.sql("""
    SELECT
        COUNT(*) AS nb_vols_avec_retard_renseigne,
        AVG(arrival_delay) AS moyenne_retard_min,
        STDDEV(arrival_delay) AS ecart_type_min,
        SKEWNESS(arrival_delay) AS asymetrie,
        PERCENTILE_APPROX(arrival_delay, 0.25) AS p25,
        PERCENTILE_APPROX(arrival_delay, 0.50) AS p50_mediane,
        PERCENTILE_APPROX(arrival_delay, 0.75) AS p75,
        PERCENTILE_APPROX(arrival_delay, 0.90) AS p90,
        PERCENTILE_APPROX(arrival_delay, 0.95) AS p95,
        MAX(arrival_delay) AS retard_max_min
    FROM v_silver
    WHERE arrival_delay IS NOT NULL
""")

df_dist_summary.show(truncate=False)

+-----------------------------+------------------+-----------------+----------------+-----+-----------+---+----+----+--------------+
|nb_vols_avec_retard_renseigne|moyenne_retard_min|ecart_type_min   |asymetrie       |p25  |p50_mediane|p75|p90 |p95 |retard_max_min|
+-----------------------------+------------------+-----------------+----------------+-----+-----------+---+----+----+--------------+
|5714008                      |4.407057357987598 |39.27129709388631|6.50289450478509|-13.0|-5.0       |8.0|34.0|66.0|1971.0        |
+-----------------------------+------------------+-----------------+----------------+-----+-----------+---+----+----+--------------+



**Votre justification statistique *(À compléter)* :**
> `le coefficient de skewness vaut environ 6,50. La plupart des vols arrivent à l’heure ou en avance — la médiane est de −5 minutes — mais quelques retards très longs tirent la moyenne à +4,41 minutes. Le maximum atteint 1 971 minutes, ce qui contribue probablement à l’écart-type élevé (39,27 minutes).

L’IQR est de 23 minutes (P75 − P25 = 8 − (−13)) : la moitié centrale des vols se situe entre 13 minutes d’avance et 8 minutes de retard. Pour représenter le retard d’un vol typique dans le dashboard, la médiane est donc plus pertinente que la moyenne. On peut la compléter par le P90 (34 minutes) et le P95 (66 minutes) pour rendre visibles les retards importants.`

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [ ]:
# Le Pareto indique ainsi quelle part du total des minutes de retard
#  est cumulée en ajoutant les compagnies classées de la plus
# contributrice à la moins contributrice.

GOLD_PARETO_PATH = "/lakehouse/gold/retards_par_compagnie"

df_gold_1 = spark.sql("""
    WITH stats_compagnies AS (
        SELECT
            airline,
            COUNT(*) AS nb_vols,
            SUM(GREATEST(arrival_delay, 0)) AS metric_total,
            PERCENTILE_APPROX(arrival_delay, 0.50) AS retard_mediane_min,
            PERCENTILE_APPROX(arrival_delay, 0.25) AS p25_retard_min,
            PERCENTILE_APPROX(arrival_delay, 0.75) AS p75_retard_min
        FROM v_silver
        WHERE airline IS NOT NULL
          AND arrival_delay IS NOT NULL
        GROUP BY airline
    )
    SELECT
        airline,
        nb_vols,
        metric_total AS total_retard_positif_min,
        retard_mediane_min,
        p75_retard_min - p25_retard_min AS iqr_retard_min,
        DENSE_RANK() OVER (ORDER BY metric_total DESC) AS rang_retard_total,
        ROUND(
            100.0 * SUM(metric_total) OVER (
                ORDER BY metric_total DESC, airline
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) / SUM(metric_total) OVER (),
            2
        ) AS pct_cumule_retards
    FROM stats_compagnies
""")

df_gold_1.coalesce(1) \
    .write.format("delta") \
    .mode("overwrite") \
    .save(GOLD_PARETO_PATH)

df_gold_1.show(15, truncate=False)

26/11/19 09:25:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:25:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:25:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:25:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:25:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:25:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 0

+-------+-------+------------------------+------------------+--------------+-----------------+------------------+
|airline|nb_vols|total_retard_positif_min|retard_mediane_min|iqr_retard_min|rang_retard_total|pct_cumule_retards|
+-------+-------+------------------------+------------------+--------------+-----------------+------------------+
|WN     |1242403|1.3849257E7             |-4.0              |20.0          |1                |20.04             |
|AA     |712935 |8611910.0               |-6.0              |22.0          |2                |32.5              |
|DL     |870275 |8046301.0               |-8.0              |18.0          |3                |44.15             |
|EV     |554752 |7504821.0               |-4.0              |21.0          |4                |55.01             |
|UA     |507762 |7301833.0               |-6.0              |25.0          |5                |65.57             |
|OO     |576814 |7215186.0               |-4.0              |20.0          |6           

26/11/19 09:25:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:25:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:25:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:25:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:25:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:25:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [ ]:
# La variation en pourcentage est laissée à NULL lorsqu’il n’y a pas
# de mois précédent ou que son total de retards vaut zéro, car le
# calcul ne serait pas défini. Exécute cette cellule après celle
# qui crée v_silver


GOLD_TIME_PATH = "/lakehouse/gold/retards_mensuels"

df_gold_2 = spark.sql("""
    WITH mensuel AS (
        SELECT
            DATE_TRUNC('MONTH', flight_date) AS periode,
            COUNT(*) AS nb_vols,
            COUNT(arrival_delay) AS nb_vols_avec_retard_renseigne,
            SUM(
                CASE
                    WHEN arrival_delay > 0 THEN arrival_delay
                    ELSE 0
                END
            ) AS total_retard_positif_min
        FROM v_silver
        GROUP BY DATE_TRUNC('MONTH', flight_date)
    ),
    avec_mois_precedent AS (
        SELECT
            periode,
            nb_vols,
            nb_vols_avec_retard_renseigne,
            total_retard_positif_min,
            LAG(total_retard_positif_min, 1) OVER (
                ORDER BY periode
            ) AS retard_mois_precedent_min
        FROM mensuel
    )
    SELECT
        periode,
        nb_vols,
        nb_vols_avec_retard_renseigne,
        total_retard_positif_min,
        retard_mois_precedent_min,
        total_retard_positif_min - retard_mois_precedent_min
            AS variation_retard_min,
        CASE
            WHEN retard_mois_precedent_min > 0 THEN ROUND(
                100.0 * (total_retard_positif_min - retard_mois_precedent_min)
                / retard_mois_precedent_min,
                2
            )
            ELSE NULL
        END AS variation_retard_pct
    FROM avec_mois_precedent
    ORDER BY periode
""")

df_gold_2.coalesce(1) \
    .write.format("delta") \
    .mode("overwrite") \
    .save(GOLD_TIME_PATH)

df_gold_2.show(20, truncate=False)

26/11/19 09:27:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:27:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:27:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:27:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:27:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:27:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 0

+-------------------+-------+-----------------------------+------------------------+-------------------------+--------------------+--------------------+
|periode            |nb_vols|nb_vols_avec_retard_renseigne|total_retard_positif_min|retard_mois_precedent_min|variation_retard_min|variation_retard_pct|
+-------------------+-------+-----------------------------+------------------------+-------------------------+--------------------+--------------------+
|2015-01-01 00:00:00|469968 |457013                       |5997907.0               |NULL                     |NULL                |NULL                |
|2015-02-01 00:00:00|429191 |407663                       |6155399.0               |5997907.0                |157492.0            |2.63                |
|2015-03-01 00:00:00|504312 |492138                       |6039489.0               |6155399.0                |-115910.0           |-1.88               |
|2015-04-01 00:00:00|485151 |479251                       |5181590.0              

26/11/19 09:27:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:27:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:27:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 09:27:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [6]:
GOLD_TOP_AIRLINE_PATH = "/lakehouse/gold/top_compagnie_retards_par_aeroport"

df_gold_3 = spark.sql("""
    WITH retards_par_compagnie AS (
        SELECT
            origin_airport,
            airline,
            COUNT(*) AS nb_vols,
            SUM(
                CASE
                    WHEN arrival_delay > 0 THEN arrival_delay
                    ELSE 0
                END
            ) AS total_retard_positif_min,
            PERCENTILE_APPROX(arrival_delay, 0.50) AS retard_median_min
        FROM v_silver
        WHERE origin_airport IS NOT NULL
          AND airline IS NOT NULL
          AND arrival_delay IS NOT NULL
        GROUP BY origin_airport, airline
    ),
    classement AS (
        SELECT
            origin_airport,
            airline,
            nb_vols,
            total_retard_positif_min,
            retard_median_min,
            ROW_NUMBER() OVER (
                PARTITION BY origin_airport
                ORDER BY total_retard_positif_min DESC, nb_vols DESC, airline
            ) AS rang
        FROM retards_par_compagnie
    )
    SELECT
        origin_airport,
        airline,
        nb_vols,
        total_retard_positif_min,
        retard_median_min
    FROM classement
    WHERE rang = 1
""")

df_gold_3.coalesce(1) \
    .write.format("delta") \
    .mode("overwrite") \
    .save(GOLD_TOP_AIRLINE_PATH)

df_gold_3.show(truncate=False)

spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)

+--------------+-------+-------+------------------------+-----------------+
|origin_airport|airline|nb_vols|total_retard_positif_min|retard_median_min|
+--------------+-------+-------+------------------------+-----------------+
|10135         |EV     |208    |2987.0                  |-3.0             |
|10136         |MQ     |192    |1606.0                  |-16.0            |
|10140         |WN     |1003   |8847.0                  |-3.0             |
|10141         |OO     |70     |451.0                   |-4.0             |
|10146         |EV     |86     |1267.0                  |-2.0             |
|10154         |B6     |28     |863.0                   |-2.0             |
|10155         |EV     |141    |362.0                   |-16.0            |
|10157         |OO     |119    |1985.0                  |-7.0             |
|10158         |NK     |245    |1120.0                  |-9.0             |
|10165         |AS     |9      |206.0                   |17.0             |
|10170      

+------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                  |metrics                                                                                                                                                                                             |
+------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hdfs://namenode:9000/lakehouse/silver/flights_enriched|{1, 14, {189311828, 189311828, 1.89311828E8, 1, 189311828}, {12798042, 14002247, 1.33676785E7, 14, 187147499}, 1, NULL, NULL, 1, 1, 14, 0, false, 0, 0, 1795080669260, 0, 10, 0, NULL,

OPTIMIZE regroupe les petits fichiers de la table Delta en fichiers plus grands. Sur HDFS, chaque fichier et chaque bloc ajoute des métadonnées gérées par le NameNode. Une trop grande quantité de petits fichiers alourdit donc ces métadonnées et augmente le coût des opérations de lecture. La compaction réduit ce nombre de fichiers et facilite les lectures.